## Paper Auto-Classifier 📚
### 1) Importar Librerias 

In [1]:
#Asegurate de tener activado el entorno de programación con el interprete python 3.11
# (En terminal) 
# pip install langchain_huggingface
# pip install tqdm

import pandas as pd 
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

e:\LLMzCor\env_LLMzCor_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2) Carga BD de Bacterias 🦠🧫👨‍🔬👩‍🔬

In [34]:
#Modifica la ruta de la base de datos
#Selecona la pesataña del excel
file_path = r"E:\LLMzCor\LLMzCor.github.io\Test\Test_files\test_file_B.xlsx"
# sheet = "Test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    #'Estado',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary',
    # 'Publication Year',
    # 'Journal/Book',
    # 'Alerta'
]
df = pd.read_excel(
    io=file_path,
    # sheet_name=sheet,
    usecols=usecols, index_col=0
)

In [35]:
#df-unfilter.head()
df.shape

(1145, 6)

#### Aplica filtros de Excluir (Opcional)

In [36]:
#Filtro los excluidos
# df = df[df['Estado']!="Excluir"]

df_shape = df.shape

#Calculo cuantos paper se excluyeron
resta = df .shape[0]- df.shape[0]
print("df Shape ", df_shape, " depués del filtro ", resta)

df.head()

df Shape  (1145, 6)  depués del filtro  0


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
34485172,Pharmacokinetics and Pharmacodynamics of a Nov...,Phage therapy is one of the most promising alt...,1,1,0,Phage øKp_Pokalde_002 is a potent therapeutic ...
34228538,Enhanced Antibacterial Activity of Repurposed ...,Klebsiella pneumoniae is an opportunistic Gram...,1,1,0,A lytic phage-mitomycin C combination was effe...
34199531,Dehydroabietic Acid Microencapsulation Potenti...,The antimicrobial activity of dehydroabietic a...,0,1,0,The obtained results indicate that DHA is a pr...
35663465,Airway Epithelial Cells Differentially Adapt T...,Background: Pneumonia is often elicited by bac...,0,0,0,Study of type-2-alveolar-epithelial-cells inmu...
30266769,Human Factor H Domains 6 and 7 Fused to IgG1 F...,Novel therapeutics against multidrug-resistant...,0,1,0,Human Factor H Domains 6 and 7 Fused to IgG1 F...


#### _____________2.b) Carga de BD de Tratamientos ⚗️🧪👨‍🔬(Opcional)

In [ ]:
#Caraga la base de datos de los tratamientos (tx) para cada bacteria
treatment_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Treatment_2017.xlsx"
tx_df = pd.read_excel(io= treatment_path, index_col=0)
#tx_df.head()

#Selecciona los tratamiento de primera eleccion utilizados hasta 2017
Bacteria = "Chlamydia_trachomatis" 
treatment= tx_df.loc[Bacteria,"First-line treatment until 2017"]
print(treatment)

### 3) LLM Funciones ⚠️ Importante Ejecutar‼️

In [37]:
clasificador=Clasificador()

In [38]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Title"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Title"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

### _______________________Prueba clasificando un solo paper (Opcional)

In [15]:

#paper=df.loc[22304240, "Abstract"]
#paper=df.loc[22525317, "Title"]
#clasificador.clasificacion(paper)

### 4) Ask LLM to classify df 

---
Abreviatura dataframes de bacterias:
+ df --> Chlamydia trachomatis
+ df_Cd --> Clostridium difficile
+ df_Hi --> Haemopilus influenzae
+ df_Kp --> Klebsiella pneumoniae
+ df_Ng --> Neisseria gonorrhoeae
+ df_Sh --> Shigela spp
+ df_Rk --> Ricketttsia
---

###  4) Partición del df (Opcional) 🚧🚩Crhistian ver aquí 🚩🚧
Aunque df_ptit le pasa 10 o20  papers nos devuelve solo 9

### 4a) Ask_llm(df) Data Frame sin particionar

In [39]:
df_Ct = ask_llm(df)
shape_df= df_Ct.shape


print( " Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a novel virulent Klebsiella phage, which implies a potential new treatment for Klebsiella infections.'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,

Se corto el proceso del LLM por este motivo : list index out of range en el id 30787927
 Shape df_classified  (10, 8)


### 4b) Ask df_ptits Particionado

In [48]:
df_ptit=df.iloc[:10,:]
shape_df_ptit= df_ptit.shape

df_classified_1 = ask_llm(df_ptit)
shape_df= df_classified_1.shape
print("--------------------------------------------------------")

print(" The df_ptit shape is ", shape_df_ptit, ", The df_classified shape is """, shape_df )
df_classified_1[0:10][["Title","ai_label","ai_summary"]]

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a novel virulent Klebsiella phage, which implies a potential new treatment for Klebsiella infections.'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,

--------------------------------------------------------
 The df_ptit shape is  (10, 6) , The df_classified shape is  (10, 8)


,Title,ai_label,ai_summary
PMID,,,
34485172,Pharmacokinetics and Pharmacodynamics of a Nov...,2,'The paper discusses a novel virulent Klebsie...
34228538,Enhanced Antibacterial Activity of Repurposed ...,2,'The paper discusses an enhanced antibacteria...
34199531,Dehydroabietic Acid Microencapsulation Potenti...,2,'The paper discusses the potential of Dehydro...
35663465,Airway Epithelial Cells Differentially Adapt T...,4,'The paper does not discuss multiresistance b...
30266769,Human Factor H Domains 6 and 7 Fused to IgG1 F...,2,'The paper discusses a new immunotherapeutic ...
35067598,Molecular Surveillance and Prediction of Antim...,1,'The paper discusses molecular surveillance a...
31654792,Is there a future for the ongoing use of azith...,2,'The paper discusses the future use of azithr...
31344102,Resistance of Neisseria gonorrhoeae isolates t...,1,'The abstract discusses the resistance of Nei...
29776806,Resistant gonorrhoea: east meets west,4,'The abstract does not discuss multiresistanc...


In [41]:
df_ptit=df.iloc[10:20,:]
shape_df_ptit= df_ptit.shape

df_classified_2 = ask_llm(df_ptit)
shape_df= df_classified_2.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : list index out of range en el id 30787927
Shape df_ptit  (10, 6) , Shape df_classified  (0, 8)


In [42]:
df_ptit=df.iloc[20:30,:]
shape_df_ptit= df_ptit.shape

df_classified_3 = ask_llm(df_ptit)
shape_df= df_classified_3.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )
df_classified_3

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a strain of Neisseria gonorrhoeae that has developed resistance to both ceftriaxone and high-level azithromycin, which are commonly used treatments for gonorrhoea, thus it falls under the category of multire

Se corto el proceso del LLM por este motivo : list index out of range en el id 27389169
Shape df_ptit  (10, 6) , Shape df_classified  (1, 8)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
29991383,Gonorrhoea treatment failure caused by a Neiss...,We describe a gonorrhoea case with combined hi...,1,0,0,"Creo que está mal catalogado, es más un caso d...",1,'The paper discusses a strain of Neisseria go...


In [49]:
df_ptit=df.iloc[30:40,:]
shape_df_ptit= df_ptit.shape

df_classified_4 = ask_llm(df_ptit)
shape_df= df_classified_4.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a new treatment for infections caused by multidrug-resistant Klebsiella pneumoniae using topically applied bacteriophage in a rat model.'' has dtype incompatible with float64, please explicitly cast to a com

Se corto el proceso del LLM por este motivo : list index out of range en el id 26131720
Shape df_ptit  (10, 6) , Shape df_classified  (3, 8)


In [50]:
df_ptit=df.iloc[40:50,:]
shape_df_ptit= df_ptit.shape

df_classified_5 = ask_llm(df_ptit)
shape_df= df_classified_5.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the modulatory effects of caffeine and pentoxifylline on aromatic antibiotics, suggesting a potential role in enhancing antibiotic efficacy, which falls under the category of New Treatments.'' has dtype inco

Se corto el proceso del LLM por este motivo : list index out of range en el id 34516580
Shape df_ptit  (10, 6) , Shape df_classified  (2, 8)


In [51]:
df_ptit=df.iloc[50:60,:]
shape_df_ptit= df_ptit.shape

df_classified_6 = ask_llm(df_ptit)
shape_df= df_classified_6.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the in-vitro activity of several new or comparative antimicrobial agents, including cefiderocol, cefepime/zidebactam, cefepime/enmetazobactam, omadacycline, and eravacycline, against carbapenem-nonsusceptibl

Shape df_ptit  (10, 6) , Shape df_classified  (10, 8)


In [52]:
df_ptit=df.iloc[60:70,:]
shape_df_ptit= df_ptit.shape

df_classified_7 = ask_llm(df_ptit)
shape_df= df_classified_7.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses ceftriaxone resistance in Neisseria gonorrhoeae, which falls under the category of multiresistance bacteria stains report. It does not discuss new treatments or immunization.'' has dtype incompatible with fl

Se corto el proceso del LLM por este motivo : list index out of range en el id 28931240
Shape df_ptit  (10, 6) , Shape df_classified  (4, 8)


In [53]:
df_ptit=df.iloc[70:80,:]
shape_df_ptit= df_ptit.shape

df_classified_8 = ask_llm(df_ptit)
shape_df= df_classified_8.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss multiresistance bacteria stains report, new treatments, nor immunization. Instead, it focuses on the prevalence and dominance of specific variants of Chlamydia trachomatis L serovars in the U.S.A.'' h

Se corto el proceso del LLM por este motivo : list index out of range en el id 26089520
Shape df_ptit  (10, 6) , Shape df_classified  (2, 8)


In [54]:
df_ptit=df.iloc[80:90,:]
shape_df_ptit= df_ptit.shape

df_classified_9 = ask_llm(df_ptit)
shape_df= df_classified_9.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a potential new treatment for Neisseria gonorrhoeae infections, focusing on the use of the C4BP-IgM protein as a therapeutic approach.'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]


Shape df_ptit  (10, 6) , Shape df_classified  (10, 8)


In [55]:
df_ptit=df.iloc[90:100,:]
shape_df_ptit= df_ptit.shape

df_classified_10 = ask_llm(df_ptit)
shape_df= df_classified_10.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract discusses the characterization of azithromycin-resistant Neisseria gonorrhoeae, which falls under the category of multidrug-resistant bacteria stains.'' has dtype incompatible with float64, please explicitly cast t

Se corto el proceso del LLM por este motivo : list index out of range en el id 24378136
Shape df_ptit  (10, 6) , Shape df_classified  (4, 8)


In [64]:
df_lista= [df_classified_1, 
             df_classified_2, 
             df_classified_3, 
             df_classified_4, 
             df_classified_5, 
             df_classified_6, 
             df_classified_8, 
             df_classified_7, 
             df_classified_9, 
             df_classified_10
            ]
df_Cta = pd.concat(df_lista, ignore_index=False)


In [65]:
print(df_Cta.shape)

df_Cta

(46, 8)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
34485172,Pharmacokinetics and Pharmacodynamics of a Nov...,Phage therapy is one of the most promising alt...,1,1,0,Phage øKp_Pokalde_002 is a potent therapeutic ...,2,'The paper discusses a novel virulent Klebsie...
34228538,Enhanced Antibacterial Activity of Repurposed ...,Klebsiella pneumoniae is an opportunistic Gram...,1,1,0,A lytic phage-mitomycin C combination was effe...,2,'The paper discusses an enhanced antibacteria...
34199531,Dehydroabietic Acid Microencapsulation Potenti...,The antimicrobial activity of dehydroabietic a...,0,1,0,The obtained results indicate that DHA is a pr...,2,'The paper discusses the potential of Dehydro...
35663465,Airway Epithelial Cells Differentially Adapt T...,Background: Pneumonia is often elicited by bac...,0,0,0,Study of type-2-alveolar-epithelial-cells inmu...,4,'The paper does not discuss multiresistance b...
30266769,Human Factor H Domains 6 and 7 Fused to IgG1 F...,Novel therapeutics against multidrug-resistant...,0,1,0,Human Factor H Domains 6 and 7 Fused to IgG1 F...,2,'The paper discusses a new immunotherapeutic ...
35067598,Molecular Surveillance and Prediction of Antim...,Background: The aims of this study was to desc...,1,0,0,"Multidrug-resistant strains Northern Alberta, ...",1,'The paper discusses molecular surveillance a...
31654792,Is there a future for the ongoing use of azith...,Is there a future for the ongoing use of azith...,0,1,0,NaN,2,'The paper discusses the future use of azithr...
31344102,Resistance of Neisseria gonorrhoeae isolates t...,The goal of this work was to study the phenoty...,1,0,0,Resistance to different drugs and the genetic ...,1,'The abstract discusses the resistance of Nei...
29776806,Resistant gonorrhoea: east meets west,Resistant gonorrhoea: east meets west,1,0,0,NaN,4,'The abstract does not discuss multiresistanc...


### ----------Prueba de automatización de clasificaciones ----------

In [66]:
# Define el tamaño de cada partición
batch_size = 10
df_list = []

# Itera sobre el DataFrame en bloques de batch_size
for start in range(0, df.shape[0], batch_size):
    end = start + batch_size
    df_ptit = df.iloc[start:end, :]
    print(f"Procesando filas {start} a {end-1}")
    df_classified = ask_llm(df_ptit)
    df_list.append(df_classified)

# Une todos los resultados en un solo DataFrame
df_Cta = pd.concat(df_list, ignore_index=False)
print("Shape final:", df_Cta.shape)
df_Cta.head()

Procesando filas 0 a 9


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a novel virulent Klebsiella phage, which implies a potential new treatment for Klebsiella infections.'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,

Procesando filas 10 a 19


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : list index out of range en el id 30787927
Procesando filas 20 a 29


...

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a new antimicrobial, DIS-73285, which has high in vitro activity against MDR and XDR Neisseria gonorrhoeae, indicating a new treatment.' ' has dtype incompatible with float64, please explicitly cast to a com

Procesando filas 1140 a 1149


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_22668\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : list index out of range en el id 36312943
Shape final: (584, 8)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
34485172,Pharmacokinetics and Pharmacodynamics of a Nov...,Phage therapy is one of the most promising alt...,1,1,0,Phage øKp_Pokalde_002 is a potent therapeutic ...,2,'The paper discusses a novel virulent Klebsie...
34228538,Enhanced Antibacterial Activity of Repurposed ...,Klebsiella pneumoniae is an opportunistic Gram...,1,1,0,A lytic phage-mitomycin C combination was effe...,2,'The paper discusses an enhanced antibacteria...
34199531,Dehydroabietic Acid Microencapsulation Potenti...,The antimicrobial activity of dehydroabietic a...,0,1,0,The obtained results indicate that DHA is a pr...,2,'The paper discusses the potential of Dehydro...
35663465,Airway Epithelial Cells Differentially Adapt T...,Background: Pneumonia is often elicited by bac...,0,0,0,Study of type-2-alveolar-epithelial-cells inmu...,4,'The paper does not discuss multiresistance b...
30266769,Human Factor H Domains 6 and 7 Fused to IgG1 F...,Novel therapeutics against multidrug-resistant...,0,1,0,Human Factor H Domains 6 and 7 Fused to IgG1 F...,2,'The paper discusses a new immunotherapeutic ...


### 5) Evaluar 

In [67]:
evaluacion_score(df_Cta)

values=[1 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 0 1]
ai_opcion2
values=[1 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion3
values=[1 1 0]
ai_opcion0
values=[0 1 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion3
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[1 1 

0.8287671232876712

# 6) Limpieza de Df

In [69]:
columns_excluded = ['1) Antimicrobial Resistance stain', 
                    '2) New treatment', 
                    '3) Immunization', 
                    'Abstract', 
                    'Human_summary'
                    ]

In [70]:
df_clean= df_Cta.drop(columns= columns_excluded)

In [71]:
df_clean

,Title,ai_label,ai_summary
PMID,,,
34485172,Pharmacokinetics and Pharmacodynamics of a Nov...,2,'The paper discusses a novel virulent Klebsie...
34228538,Enhanced Antibacterial Activity of Repurposed ...,2,'The paper discusses an enhanced antibacteria...
34199531,Dehydroabietic Acid Microencapsulation Potenti...,2,'The paper discusses the potential of Dehydro...
35663465,Airway Epithelial Cells Differentially Adapt T...,4,'The paper does not discuss multiresistance b...
30266769,Human Factor H Domains 6 and 7 Fused to IgG1 F...,2,'The paper discusses a new immunotherapeutic ...
...,...,...,...
35633723,Identification and Comparative Genomic Analysi...,4,'The paper focuses on the identification and ...
34958740,Whole-genome analysis of haemophilus influenza...,4,'The abstract does not discuss multiresistanc...
26700635,Outbreak of a beta-lactam resistant non-typeab...,1,'The abstract discusses a specific strain of ...


# EDA

In [72]:
print("Se analizaron ", df_clean.shape[0], " papers.")

Se analizaron  584  papers.


#### AI Label:

1) Antimicrobial Resistance stain
2) New treatment
3) Immunization
4) None

In [73]:
df_clean['ai_label'].value_counts()

ai_label
1    271
2    215
4     73
3     25
Name: count, dtype: int64